# 🤖 AI Agent Builder
### 🛠️ Developed by: **Arafat Ahmed Mubin**
### 🌐 Brand: **2M**
---
Design, test, and deploy custom AI Agents via **OpenRouter** (free models included).
Compatible with **Google Colab** and **Kaggle**.

## 🛠️ Step 1: Environment Setup

In [ ]:
import os
try:
    import google.colab
    ENV = "Colab"
except ImportError:
    ENV = "Kaggle" if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") else "Local"
print(f"✅ Detected: {ENV}")
if ENV == "Kaggle":
    print("ℹ️  Ensure 'Internet' is ON in Kaggle Notebook Settings.")
!pip install -q -U openai gradio
print("✅ Packages ready.")

## 🧠 Step 2: Agent Architecture

In [ ]:
from openai import OpenAI
import gradio as gr, json, base64

# ── Free models on OpenRouter ─────────────────────────────────────────────────
MODELS = {
    "Meta Llama 3.1 8B (Free)": "meta-llama/llama-3.1-8b-instruct:free",
    "Mistral 7B (Free)"        : "mistralai/mistral-7b-instruct:free",
    "Gemma 2 9B (Free)"        : "google/gemma-2-9b-it:free",
    "Phi-3 Mini (Free)"        : "microsoft/phi-3-mini-128k-instruct:free",
    "Auto (Best Available)"    : "openrouter/auto",
}

TEMPLATES = {
    "Professional Coder"  : "You are a Senior Software Architect. Provide clean, efficient, well-documented code. Always explain your design decisions.",
    "SEO & Content Master" : "You are a world-class SEO expert and copywriter. Produce engaging, high-ranking content with proper keyword placement.",
    "Creative Storyteller" : "You are a master novelist. Craft vivid, emotionally resonant stories with strong characters and compelling plot arcs.",
    "Math & Logic Analyst" : "You are a Mathematical Genius. Solve problems step-by-step, explain your reasoning clearly.",
    "Research Assistant"   : "You are an expert research assistant. Provide well-sourced, balanced, and comprehensive answers.",
    "Custom"               : "You are a helpful AI assistant."
}

def get_client(api_key):
    if not api_key.strip(): return None
    return OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)

def chat_with_agent(api_key, model_choice, system_prompt, temperature, max_tokens, message, history):
    client = get_client(api_key)
    if not client:
        return history + [[message, "❌ Enter a valid OpenRouter API Key in the Setup tab."]], ""
    if not message.strip():
        return history, ""
    try:
        model_id = MODELS.get(model_choice, "openrouter/auto")
        messages  = [{"role": "system", "content": system_prompt}]
        for user_msg, bot_msg in history:
            messages.append({"role": "user",      "content": user_msg})
            messages.append({"role": "assistant", "content": bot_msg})
        messages.append({"role": "user", "content": message})

        response = client.chat.completions.create(
            model=model_id,
            messages=messages,
            temperature=float(temperature),
            max_tokens=int(max_tokens)
        )
        reply = response.choices[0].message.content
        history.append([message, reply])
        return history, ""
    except Exception as e:
        history.append([message, f"⚠️ API Error: {e}"])
        return history, ""

def export_config(prompt, temp, model):
    data = {"system_prompt": prompt, "temperature": temp, "model": model}
    return "AGENT_CFG_" + base64.b64encode(json.dumps(data).encode()).decode()

print("✅ Agent architecture ready.")

## 🏗️ Step 3: Agent Builder Interface

In [ ]:
with gr.Blocks(theme=gr.themes.Soft(), title="2M AI Agent Builder") as builder:
    gr.Markdown("# 🤖 2M AI Agent Builder Studio\n**OpenRouter · Free LLMs · Custom Personas · Colab & Kaggle**")

    api_key_state = gr.State("")

    with gr.Tab("🔑 Setup"):
        gr.Markdown("### 1. Get your free API key from [openrouter.ai/keys](https://openrouter.ai/keys)")
        key_input    = gr.Textbox(label="OpenRouter API Key", type="password", placeholder="sk-or-v1-...")
        save_btn     = gr.Button("💾 Save & Test Connection", variant="primary")
        setup_status = gr.Markdown("**Status:** 🔴 Not connected")

        def validate_key(k):
            k = k.strip()
            if k.startswith("sk-or"):
                return k, "**Status:** 🟢 Connected — ready to chat!"
            return "", "**Status:** ⚠️ Invalid key format (should start with `sk-or`)."

        save_btn.click(validate_key, key_input, [api_key_state, setup_status])

    with gr.Tab("🎨 Designer"):
        gr.Markdown("### Configure your agent persona and parameters.")
        template_drop = gr.Dropdown(list(TEMPLATES.keys()), label="Persona Template", value="Professional Coder")
        model_drop    = gr.Dropdown(list(MODELS.keys()), label="Language Model", value="Meta Llama 3.1 8B (Free)")
        system_prompt = gr.Textbox(label="System Prompt", lines=8, value=TEMPLATES["Professional Coder"])
        with gr.Row():
            temp_sl      = gr.Slider(0.0, 2.0, value=0.7, step=0.05, label="Temperature (creativity)")
            max_tok_sl   = gr.Slider(256, 4096, value=1024, step=128, label="Max Tokens")
        template_drop.change(lambda t: TEMPLATES[t], template_drop, system_prompt)
        export_btn  = gr.Button("📤 Export Config Code")
        export_code = gr.Textbox(label="Shareable Config", interactive=False)
        export_btn.click(export_config, [system_prompt, temp_sl, model_drop], export_code)

    with gr.Tab("💬 Live Chat"):
        chatbot   = gr.Chatbot(label="Conversation", height=500)
        msg_input = gr.Textbox(label="Your message", placeholder="Ask your agent anything...", lines=2)
        with gr.Row():
            send_btn  = gr.Button("📤 Send", variant="primary")
            clear_btn = gr.Button("🗑️ Clear Chat")

        send_btn.click(
            chat_with_agent,
            [api_key_state, model_drop, system_prompt, temp_sl, max_tok_sl, msg_input, chatbot],
            [chatbot, msg_input]
        )
        msg_input.submit(
            chat_with_agent,
            [api_key_state, model_drop, system_prompt, temp_sl, max_tok_sl, msg_input, chatbot],
            [chatbot, msg_input]
        )
        clear_btn.click(lambda: ([], ""), outputs=[chatbot, msg_input])

builder.launch(share=True, debug=False)

---
**© 2026 2M | All Rights Reserved**  
*Powered by the 2M Ecosystem (2M Dev's · 2M Future Facts · 2M Business Blogs)*